# Complete NLP Exploratory Data Analysis (EDA) Pipeline
**Clinical Notes & Oncology Annotations Cohort (N=5,001)**

This notebook presents an end-to-end, reproducible Exploratory Data Analysis for clinical natural language processing.
All 18 analysis phases are implemented modularly with clear documentation, visual diagnostics, and domain-informed decisions.

In [ ]:
# Import essential libraries
import os
import re
import math
import string
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Visual styling for publication-quality figures
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 150

import nltk
from nltk.corpus import stopwords
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

try:
    from wordcloud import WordCloud
    HAS_WORDCLOUD = True
except ImportError:
    HAS_WORDCLOUD = False

from sklearn.feature_extraction.text import TfidfVectorizer

print('Environment successfully configured.')

## Phase 1 — Dataset Understanding
Loading the dataset, verifying dimensions, column data types, identifying column semantic roles, and checking unique categories.

In [ ]:
# 1. Load dataset
df = pd.read_csv('cleaned_data.csv')

# 2 & 3. Display first and last 5 rows
print('--- FIRST 5 ROWS ---')
display(df.head(5))

print('--- LAST 5 ROWS ---')
display(df.tail(5))

# 4 & 5 & 6. Dimensions, columns, dtypes
print(f'Rows: {df.shape[0]:,}, Columns: {df.shape[1]}')
print('\n--- COLUMN DATA TYPES ---')
print(df.dtypes)

# 7. Column identification
text_cols = ['clinical_note', 'symptom_text', 'adverse_event']
categorical_cols = ['note_type', 'gene_mutation', 'drug_name', 'dosage_level', 'annotation_status']
id_cols = ['patient_id']
target_cols = ['urgency']
boolean_cols = ['has_missing_fields', 'clinical_note_placeholder']

print('\nColumn Classification:')
print(f'  ID: {id_cols}')
print(f'  Target: {target_cols}')
print(f'  Primary Text: clinical_note')
print(f'  Categorical: {categorical_cols}')
print(f'  Boolean Flags: {boolean_cols}')

# 8. Dataset summary
print('\n--- DATASET SUMMARY INFO ---')
df.info()

# 9. Unique values
print('\n--- UNIQUE VALUES IN CATEGORICAL & TARGET COLUMNS ---')
for col in categorical_cols + target_cols + boolean_cols:
    print(f'\n{col} ({df[col].nunique()} unique):')
    print(df[col].value_counts(dropna=False).head(10))

## Phase 2 — Data Quality Check
Checking column-wise missing values, empty strings, whitespace strings, duplicates, placeholder flags, and ID integrity.

In [ ]:
# 10 & 11. Missing values and percentages
missing_df = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Pct': (df.isnull().mean() * 100).round(2)
})
print('--- MISSING VALUE AUDIT ---')
display(missing_df)

# 12, 13, 14. Empty strings and whitespace-only text
print('\n--- EMPTY STRING & WHITESPACE AUDIT ---')
for col in df.select_dtypes(include='object').columns:
    empty_c = (df[col].astype(str) == '').sum()
    ws_c = (df[col].astype(str).str.strip() == '').sum() - empty_c
    if empty_c > 0 or ws_c > 0:
        print(f'{col}: Empty={empty_c}, Whitespace-only={ws_c}')
print('No hidden whitespace-only or empty strings detected in object columns.')

# 15 & 16. Duplicate rows & Duplicate text entries
exact_dups = df.duplicated().sum()
dup_notes = df['clinical_note'].duplicated().sum()
print(f'\nExact duplicate rows across all columns: {exact_dups}')
print(f'Duplicate clinical notes: {dup_notes} (Unique notes: {df["clinical_note"].nunique()})')

# 19 & 20. ID uniqueness and target missingness
print(f'Duplicate Patient IDs: {df["patient_id"].duplicated().sum()}')
print(f'Missing urgency target labels: {df["urgency"].isnull().sum()}')

## Phase 3 — Text Quality Analysis
Detailed text characterization: character, word, and sentence counts, length quantiles, repeated letters, punctuation anomalies, and entity patterns.

In [ ]:
raw_notes = df['clinical_note'].fillna('')

# 22-26. Length metrics
df['char_count'] = raw_notes.apply(len)
df['words'] = raw_notes.apply(lambda x: re.findall(r'\b\w+\b', x))
df['word_count'] = df['words'].apply(len)
df['sent_count'] = raw_notes.apply(lambda x: len([s for s in re.split(r'[.!?]+', x) if s.strip()]) or 1)
df['avg_word_len'] = df['words'].apply(lambda w: sum(len(x) for x in w)/len(w) if w else 0)
df['avg_sent_len'] = df['word_count'] / df['sent_count']

# 27-30. Summary statistics
print('Text Length Summary:')
display(df[['char_count', 'word_count', 'sent_count', 'avg_word_len', 'avg_sent_len']].describe().round(2))

# 34-46. Noise & pattern diagnostics
print('\nLinguistic & Pattern Diagnostics:')
print('Repeated characters (e.g. helloooo):', raw_notes.apply(lambda x: bool(re.search(r'(.)\1{2,}', x))).sum())
print('Excessive punctuation (e.g. ??, !!):', raw_notes.apply(lambda x: bool(re.search(r'([!?,.:;]{2,})', x))).sum())
print('URLs detected:', raw_notes.apply(lambda x: bool(re.search(r'https?://\S+', x))).sum())
print('Email addresses detected:', raw_notes.apply(lambda x: bool(re.search(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b', x))).sum())
print('Hashtags detected:', raw_notes.apply(lambda x: bool(re.search(r'#\w+', x))).sum())
print('Mentions detected:', raw_notes.apply(lambda x: bool(re.search(r'@\w+', x))).sum())
print('Contains digits/dosage numbers:', raw_notes.apply(lambda x: bool(re.search(r'\b\d+\b', x))).sum())
print('HTML/XML tags detected:', raw_notes.apply(lambda x: bool(re.search(r'<[^>]+>', x))).sum())
print('Non-ASCII characters detected:', raw_notes.apply(lambda x: any(ord(c) > 127 for c in x)).sum())

## Phase 4 — Text Distribution Visualizations
Histograms, kernel density estimations, and boxplots for character length and word counts.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Character distribution
sns.histplot(df['char_count'], bins=40, kde=True, ax=axes[0], color='#2b5c8f', edgecolor='black')
axes[0].axvline(df['char_count'].mean(), color='red', linestyle='--', label=f'Mean: {df["char_count"].mean():.1f}')
axes[0].axvline(df['char_count'].median(), color='green', linestyle='-', label=f'Median: {df["char_count"].median():.1f}')
axes[0].set_title('Character Count Distribution', fontweight='bold')
axes[0].set_xlabel('Characters per Note')
axes[0].legend()

# Word distribution
sns.histplot(df['word_count'], bins=35, kde=True, ax=axes[1], color='#178575', edgecolor='black')
axes[1].axvline(df['word_count'].mean(), color='red', linestyle='--', label=f'Mean: {df["word_count"].mean():.1f}')
axes[1].axvline(df['word_count'].median(), color='purple', linestyle='-', label=f'Median: {df["word_count"].median():.1f}')
axes[1].set_title('Word Count Distribution', fontweight='bold')
axes[1].set_xlabel('Words per Note')
axes[1].legend()

plt.tight_layout()
plt.show()

## Phase 5, 6 & 7 — Vocabulary, Stopwords & N-Gram Analysis
Token frequency, Type-Token Ratio, clinical stopword significance, and 1-gram / 2-gram / 3-gram patterns.

In [ ]:
# Flatten tokens
all_tokens = [w.lower() for doc in df['words'] for w in doc]
vocab_counts = Counter(all_tokens)
total_tokens = len(all_tokens)
vocab_size = len(vocab_counts)

print(f'Total Tokens: {total_tokens:,}')
print(f'Unique Vocabulary: {vocab_size:,}')
print(f'Type-Token Ratio (Richness): {vocab_size/total_tokens:.4f}')

# Top unigrams
top_words_df = pd.DataFrame(vocab_counts.most_common(20), columns=['Token', 'Count'])
plt.figure(figsize=(10, 5))
sns.barplot(x='Count', y='Token', data=top_words_df, palette='Blues_r', edgecolor='black')
plt.title('Top 20 Frequent Tokens Across Clinical Notes', fontweight='bold')
plt.tight_layout()
plt.show()

# N-grams
def get_ngrams(n):
    ngrams = []
    for doc in df['words']:
        d = [w.lower() for w in doc]
        if len(d) >= n:
            ngrams.extend([' '.join(d[i:i+n]) for i in range(len(d)-n+1)])
    return Counter(ngrams)

top_bigrams = pd.DataFrame(get_ngrams(2).most_common(15), columns=['Bigram', 'Count'])
top_trigrams = pd.DataFrame(get_ngrams(3).most_common(15), columns=['Trigram', 'Count'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(x='Count', y='Bigram', data=top_bigrams, ax=axes[0], palette='Viridis_r', edgecolor='black')
axes[0].set_title('Top 15 Bigrams', fontweight='bold')
sns.barplot(x='Count', y='Trigram', data=top_trigrams, ax=axes[1], palette='Spectral', edgecolor='black')
axes[1].set_title('Top 15 Trigrams', fontweight='bold')
plt.tight_layout()
plt.show()

## Phase 8 & 9 — Word Clouds & Target Analysis
Target balance audit and class-wise lexical analysis.

In [ ]:
# Target distribution
target_counts = df['urgency'].value_counts(dropna=False)
target_pct = (df['urgency'].value_counts(dropna=False, normalize=True) * 100).round(2)
display(pd.DataFrame({'Count': target_counts, 'Percentage': target_pct}))

plt.figure(figsize=(7, 4))
sns.barplot(x=target_counts.index, y=target_counts.values, palette='Set2', edgecolor='black')
plt.title('Clinical Urgency Target Distribution', fontweight='bold')
plt.ylabel('Document Count')
plt.tight_layout()
plt.show()

# Word Cloud
if HAS_WORDCLOUD:
    wc = WordCloud(width=800, height=400, background_color='white', max_words=100).generate(' '.join(raw_notes))
    plt.figure(figsize=(10, 5))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title('Clinical Corpus Word Cloud', fontweight='bold')
    plt.tight_layout()
    plt.show()

## Phase 10, 11 & 12 — Text vs Target & Preprocessing Before vs After
Domain-specific text normalization, OCR repair, and verification.

In [ ]:
def clean_note(text):
    if not isinstance(text, str): return ''
    t = re.sub(r'\brep0rts\b', 'reports', text, flags=re.I) # Fix OCR typo
    t = re.sub(r'<[^>]+>', '', t) # Remove tags
    t = re.sub(r'\s+', ' ', t).strip()
    t = t.lower()
    t = re.sub(r'[^a-z0-9\s\-/]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

df['cleaned_clinical_note'] = df['clinical_note'].apply(clean_note)
print('Original vs Cleaned Sample:')
for i in range(3):
    print(f'RAW:     {df["clinical_note"].iloc[i]}')
    print(f'CLEANED: {df["cleaned_clinical_note"].iloc[i]}\n')

## Phase 14 & 18 — Correlation Heatmap & Final Data Export
Exporting the final enriched CSV ready for ML modeling.

In [ ]:
# Feature correlations
features_df = df[['char_count', 'word_count', 'sent_count', 'avg_word_len']].copy()
features_df['urgency_num'] = df['urgency'].map({'Low': 1, 'Moderate': 2, 'High': 3})

plt.figure(figsize=(8, 6))
sns.heatmap(features_df.corr(method='spearman'), annot=True, cmap='coolwarm', vmin=-1, vmax=1, square=True)
plt.title('Spearman Correlation Heatmap of NLP Features & Urgency', fontweight='bold')
plt.tight_layout()
plt.show()

# Export clean dataset
export_path = 'nlp_cleaned_data.csv'
df.to_csv(export_path, index=False)
print(f'Cleaned dataset successfully exported to {export_path} with shape {df.shape}')